<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            Agents | Foundations | Lab 4
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">Aside from the obvious (your resume of the future) this has business applications in any situation where you need an AI assistant with domain expertise and an ability to interact with the real world.
            </span>
            <span style="color:white;">
            <small><i><br>
            I created two Digital Twins  <br>
            <h3 style="color:green">Personal</h3>
            AVATAR<br>
            - deployed at <a href="https://avatar-aubi.fly.dev/">https://avatar-aubi.fly.dev/</a><br>
            - inspired by video at <a href="https://www.youtube.com/watch?v=srlhW4H-Gtg">https://www.youtube.com/</a><br>    
            - clone using <a href="https://github.com/ed-donner/avatar">https://github.com/ed-donner/avatar</a><br><br>
            Personalized following repo's README.md:<br>
            - knowledge.md from my son's linkedin <br>
            - faq.jsol built with claude's help<br>
            - pic.jpg<br>
            - styles.md<br>
            - design-system (I did not use Claude Design, at this time.) <br><br>
            Using Claude Code Pro, I asked (as Ed):<br>
            "I want to create a full design system for the aubi_avatar project as described in the SPEC.md file including designs for both the UIs, iconography, and also a profile pic for the Avatar as an Android version of my profile in knowledge/pic.jpg. The final product should include comprehensive information about UI and UX. Be sure to read the SPEC.md thoroughly."<br>
            It took Claude 1 hour and 46 minutes to complete, and successfully.<br><br>
            <h3 style="color:green">Business Application</h3>
            AI ASSISTANT for an Autobiography Publishing Company<br>
            - deployed at <a href="https://twin-publisher.onrender.com/">https://twin-publisher.onrender.com/</a><br>
            - repo at <a href="https://github.com/auroraa-lola/twin_publisher" target="_blank">https://github.com/auroraa-lola/twin_publisher</a><br><br>
            Using SQLite3 for Q&A, the Assistant:<br>
            - records only relevant questions that aren’t already in the database.<br>
            - sends contact details or unanswered questions to Pushover.<br>
            - marks relevant Q&A not in the database for human review. A small terminal tool (uv run review.py) is used by a human to approve or delete entries.
            </i></small>  
            </span>
        </td>
    </tr>
    
</table>


## The first big project - The Digital Twin

### But first: introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like Agents) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

## Heads up - a change from the videos

In the video, I deploy the twin for free to HuggingFace Spaces. HuggingFace has recently stopped supporting this for free!

There is a free alternative, and I explain it and give instructions later on in this lab.

In [ ]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
import sqlite3
from pypdf import PdfReader
import gradio as gr

In [ ]:
client = OpenAI()
MODEL = "gpt-5.4-mini"

In [ ]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
# For database
DB_PATH = "qa.db"

def get_conn():
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA journal_mode=WAL")
    return conn


def init_db():
    with get_conn() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS qa (
                id INTEGER PRIMARY KEY,
                question TEXT NOT NULL,
                answer TEXT NOT NULL,
                category TEXT,
                reviewed INTEGER NOT NULL DEFAULT 0,
                created_at TEXT DEFAULT CURRENT_TIMESTAMP
            )
        """)
        # Migrate a database created before the reviewed column existed
        cols = [r["name"] for r in conn.execute("PRAGMA table_info(qa)")]
        if "reviewed" not in cols:
            conn.execute("ALTER TABLE qa ADD COLUMN reviewed INTEGER NOT NULL DEFAULT 0")


def add_qa(question, answer, category=None):
    # LLM-written rows always start unreviewed
    with get_conn() as conn:
        conn.execute(
            "INSERT INTO qa (question, answer, category, reviewed) VALUES (?, ?, ?, 0)",
            (question, answer, category),
        )


def search_qa(text, limit=5):
    with get_conn() as conn:
        rows = conn.execute(
            "SELECT id, question, answer, reviewed FROM qa "
            "WHERE question LIKE ? OR answer LIKE ? "
            "ORDER BY reviewed DESC, id DESC LIMIT ?",
            (f"%{text}%", f"%{text}%", limit),
        ).fetchall()
    return [dict(r) for r in rows]


# ---------- Human review ----------
def list_unreviewed():
    with get_conn() as conn:
        rows = conn.execute(
            "SELECT id, question, answer FROM qa WHERE reviewed = 0 ORDER BY id"
        ).fetchall()
    return [dict(r) for r in rows]


def approve_qa(qa_id):
    with get_conn() as conn:
        conn.execute("UPDATE qa SET reviewed = 1 WHERE id = ?", (qa_id,))


def delete_qa(qa_id):
    with get_conn() as conn:
        conn.execute("DELETE FROM qa WHERE id = ?", (qa_id,))

In [ ]:
#Tool functions
def tool_search_qa(query):
    return {"results": search_qa(query)}

def tool_save_qa(question, answer, category=None):
    add_qa(question, answer, category)
    return {"status": "saved"}

def tool_record_unanswered_question(question):
    push(f"Unanswered question: {question}")
    return {"status": "recorded"}

def tool_record_user_details(email, name="not provided", notes="not provided"):
    push(f"Contact request: {name} | {email} | Notes: {notes}")
    return {"status": "recorded"}


TOOL_FUNCTIONS = {
    "search_qa": tool_search_qa,
    "save_qa": tool_save_qa,
    "record_unanswered_question": tool_record_unanswered_question,
    "record_user_details": tool_record_user_details,
}

# Tool schemas
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_qa",
            "description": "Search the Q&A database for existing answers. Always call this first for any question.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "Key words from the user's question"}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "save_qa",
            "description": "Save a question and its answer to the database after you have successfully answered a new question.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"},
                    "answer": {"type": "string"},
                    "category": {"type": "string"},
                },
                "required": ["question", "answer"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "record_unanswered_question",
            "description": "Record a question you could not answer, but only after you have already asked the user to clarify.",
            "parameters": {
                "type": "object",
                "properties": {"question": {"type": "string"}},
                "required": ["question"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "record_user_details",
            "description": "Record contact details when the user wants to get in touch.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {"type": "string"},
                    "name": {"type": "string"},
                    "notes": {"type": "string", "description": "Context from the conversation"},
                },
                "required": ["email"],
            },
        },
    },
]

## Using Python built-in globals()

Python has a dictionary that gives us access to all global functions.

Sidenote: for sure when we deploy, we will use this in a more protected way..

In [ ]:
globals()["tool_record_unanswered_question"]("this is a really hard question")

In [ ]:

def handle_tool_calls(tool_calls):
    results = []
    for call in tool_calls:
        fn = TOOL_FUNCTIONS.get(call.function.name)
        args = json.loads(call.function.arguments)
        print(f"Tool: {call.function.name} {args}", flush=True)
        result = fn(**args) if fn else {"error": "unknown tool"}
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": call.id,
        })
    return results

In [ ]:
reader = PdfReader("twin_publisher/publishing.pdf")
publishing = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        publishing += text

with open("twin_publisher/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
TWIN_SYSTEM_PROMPT = f"""

# Your role

You are a knowledgeable and friendy assistant running on a website, chatting with visitors of the website.
You represent the company who's website you are on.
You answer questions related to the company's business, background, skills and experience.

Here are the details of the company you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the AI Assistant of this company.

# Context

Here is a summary of the company's publishing profile so that you can answer questions:

{publishing}

Workflow for every question about autobiography publishing:
1. Call search_qa first,
2. If the question is exactly the same as the database, you MUST reply only with the answer in the database.
   Do not save duplicate questions.
3. Results have a "reviewed" flag. reviewed=1 entries are verified, so use them directly.
   reviewed=0 entries are unverified: use them only if they clearly fit, and be
   more cautious in how you phrase them.
4. If the database has a good answer, you MUST use it and reply. Do not save it again.
5. If the database has no good answer and you can answer confidently
   (from the company details above), you MUST call save_qa with the
   question and your answer BEFORE writing your reply to the user.
   Do not write your reply until save_qa has been called.
6. If the question is unrelated to autobiography publishing, steer the
   conversation back. Do not call save_qa.
7. If you cannot answer, ask ONE clarifying question first.
8. If you still cannot answer after the clarification, call
   record_unanswered_question and tell the user their question has been
   passed along.
9. If the user wants to get in touch, ask for their email (and name if
   they like), then call record_user_details.
"""

In [ ]:
# chat loop
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": TWIN_SYSTEM_PROMPT}] + history + [{"role": "user", "content": message}]

    while True:
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
        choice = response.choices[0]
        if choice.finish_reason == "tool_calls":
            messages.append(choice.message)
            messages.extend(handle_tool_calls(choice.message.tool_calls))
        else:
            return choice.message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

## Turning into Python modules

I've turned the code in the lab into python modules; that's a great practice to do after you've completed experiments in the Notebook.

You could put all the code above into 1 python script. But it's nicer to organize the code into different modules for different concerns, and that's what I've done:

`context.py` loads in the static data and constructs the System Prompt

`tools.py` contains all the code to manage and call tools, with their associated json

`app.py` contains the Gradio app and OpenAI call.

`styles.py` contains styles to apply to Gradio and this was entirely written by Claude Code!

You could have a stab at doing this yourself, then compare with my versions.

Then to try it out, open a terminal in Cursor:

`cd 1_foundations`  
`cd twin`  
`uv run app.py`

# STOP THE PRESS! Heads up...

As of 9-July-2026, HuggingFace has suddenly stopped allowing Gradio Apps to be deployed for free on HuggingFace Spaces.

This is quite a nasty surprise!

I expect they might reverse this decision. In the meantime, here's a free alternative: using Render.

You'll find complete instructions in [the file RENDER_INSTRUCTIONS in this directory](RENDER_INSTRUCTIONS.md)

If you don't mind paying for HuggingFace, the original instructions are below.

And also, here are instructions on my digital twin, which runs at very low cost on fly.io:  
https://edwarddonner.com/avatar

With my twin, not only can you notify me with a Push, but you can chat with the real me! Here's a video with how I made it, and instructions if you want to make it too. I started with this Career Conversations app.  
https://youtu.be/srlhW4H-Gtg

## Original Instructions with HF Spaces (no longer free)

We will deploy to HuggingFace Spaces.

Before you start: remember to update the files in the `twin` directory - your LinkedIn profile and summary.txt - so that it talks about you!

Also check that there's no README file within the twin directory. If there is one, please delete it. The deploy process creates a new README file in this directory for you.

## Deployment Part 1: HuggingFace

1. Visit https://huggingface.co and set up an account  
2. From the Avatar menu on the top right, choose Access Tokens. Choose "Create New Token". Give it WRITE permissions - it needs to have WRITE permissions! Keep a record of your new key.  
3. In the Cursor Terminal, run: `uvx hf auth login --token YOUR_TOKEN_HERE`, like `uvx hf auth login --token hf_xxxxxx`, to login at the command line with your key. Afterwards, run `uvx hf auth whoami` to check you're logged in  
4. Take your new token and add it to your .env file: `HF_TOKEN=hf_xxx` for the future

## Deployment Part 2: Push!

1. Go in to the twin directory: `cd 1_foundations` then `cd twin`
2. From the twin directory, enter: `uv run gradio deploy` 
3. Follow its instructions by selecting the default values: name it `twin`, specify app.py, choose cpu-basic as the hardware, say No to needing to supply secrets, and say "no" to github actions.  

### Deployment Part 3: Secrets

1. Go to https://huggingface.co and click your Avatar, go to your profile, select the Space
2. Go to the 3 dots menu and pick Settings
3. Scroll down to Variables and Secrets section
4. Press "New Secret" (not New Variable) and enter the name of `OPENAI_API_KEY` and the value of your key from the .env file (or use the relevant key for your LLM). Be careful to get this right!
5. Repeat for `PUSHOVER_USER` and `PUSHOVER_TOKEN` from your .env file
6. Nearer the top of the settings, click "Restart space" to restart it
7. Click on App near the top to return to the app, and after it has restarted - enjoy!

### Embedding in another site

To embed this in another website, select "Embed this space" from the three-dots menu.

### Troubleshooting

If you get a gradio error, try opening the logs (the button next to the 3-dot menu).  
Try adding more debug information particularly around your keys.

### Redploying the space

Just run `uv run gradio deploy` from the twin directory. You might need to delete the file README.md that Gradio created there if you want to name your space again.

### Deleting the space

From the 3 dots menu, select the Settings screen, and there's a Delete option at the bottom.

For more information on deployment:

https://www.gradio.app/guides/sharing-your-app#hosting-on-hf-spaces

### My Digital Twin

So I spend some time taking my Digital twin to the next level!  
Here it is:   
https://edwarddonner.com/avatar

Not only can you notify me with a Push, but you can chat with the real me! Here's a video with how I made it, and instructions if you want to make it too. I started with this Career Conversations app.  
https://youtu.be/srlhW4H-Gtg


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• First and foremost, deploy this for yourself! It's a real, valuable tool - the future resume..<br/>
            • Next, improve the resources - add better context about yourself. If you know RAG, then add a knowledge base about you.<br/>
            • Add in more tools! You could have a SQL database with common Q&A that the LLM could read and write from?<br/>
            • Bring in the Evaluator from the exercise in Day 4, and add other Agentic patterns.<br/>
            • Some students have added Telegram integration so that you can chat live with people on your site, along with your twin!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">Aside from the obvious (your resume of the future) this has business applications in any situation where you need an AI assistant with domain expertise and an ability to interact with the real world.
            </span>
        </td>
    </tr>
</table>